# core

> Claude Code API backend for FastLLM

`claude_code` connects FastLLM to the installed Claude Code CLI through `fastclaude`. `claude_mk_payload` passes the canonical `Msg` history and tool schemas to `astream`. `claude_acollect_stream` converts the resulting events to FastLLM's streaming output.

FastLLM executes the requested tools and includes their results in the next request's history. FastClaude resumes the pending tool call in a fresh process and supplies the result. Each request is stateless. The provider never issues a response id.

In [ ]:
#| default_exp core

In [ ]:
#| export
from fastcore.utils import *
from fastllm.types import *
from aidialog.msg_parts import ToolUse
from fastllm.anthropic import norm_sse_event, norm_tool_calls, norm_parts, norm_finish, norm_usage, finalize_usage, delta_index_fn, collate_raw, cost
from fastllm.streaming import mk_acollect_stream, Delta
from fasttransport.errors import APIError
from fastclaude.core import astream, unqual, SERVER_TOOLS

In [ ]:
from fastllm.chat import AsyncChat, lite_mk_func, mk_msgs, mk_tool_res_msg
from fastcore.test import *

`claude_mk_payload` accepts function schemas in both Responses and Chat Completions formats. It converts their parameters to fastclaude's `inputSchema`. Any non-`None` value for `web_search_options` enables Claude Code's native `WebSearch` and `WebFetch` tools.

Pass a user's subscription token as `oauth_token` when serving requests for that user. The payload sets `CLAUDE_CODE_OAUTH_TOKEN` in the child process's environment. Claude Code then uses that token instead of the machine's stored login.

The default `setting_sources=()` disables Claude Code's filesystem settings sources. A serving host's `CLAUDE.md`, hooks, and skills belong to the host account, not the user making the request. They should not configure that user's completion. Pass `setting_sources` explicitly when you want the run to load selected settings.

In [ ]:
#| export
def claude_mk_payload(msgs, model, stream=False, **kwargs):
    "Build `astream` inputs from a FastLLM request"
    tools = [dict(name=f['name'], description=f.get('description',''), inputSchema=f.get('parameters', {}))
        for t in (kwargs.get('tools') or []) if t.get('type') == 'function' and (f := t.get('function') or t)]
    native = SERVER_TOOLS if kwargs.get('web_search_options') is not None else ()
    payload = dict(msgs=list(msgs), model=model, system=kwargs.get('system') or '', tools=tools or None, native_tools=native,
        setting_sources=kwargs.get('setting_sources', ()))
    if key := kwargs.get('oauth_token'): payload['env'] = dict(CLAUDE_CODE_OAUTH_TOKEN=key)
    return payload

In [ ]:
def simple_add(a: int, b: int) -> int:
    "Add two numbers"
    return a + b

p = claude_mk_payload(mk_msgs(['What is 2+2?']), 'claude-sonnet-5', tools=[lite_mk_func(simple_add)])
test_eq(p['native_tools'], ())
rp = claude_mk_payload([], 'm', tools=[dict(type='function', name='py', parameters={'type':'object'})])
test_eq(rp['tools'][0]['name'], 'py')
p['tools'][0]

{'name': 'simple_add',
 'description': 'Add two numbers\n\nReturns:\n- type: integer',
 'inputSchema': {'type': 'object',
  'properties': {'a': {'description': '', 'type': 'integer'},
   'b': {'description': '', 'type': 'integer'}},
  'required': ['a', 'b']}}

In [ ]:
test_eq(claude_mk_payload([], 'm', web_search_options='l')['native_tools'], SERVER_TOOLS)
test_eq(claude_mk_payload([], 'm', oauth_token='sk-ant-oat01-x')['env'], dict(CLAUDE_CODE_OAUTH_TOKEN='sk-ant-oat01-x'))
test_eq(claude_mk_payload([], 'm')['setting_sources'], ())
test_eq(claude_mk_payload([], 'm', setting_sources=['project'])['setting_sources'], ['project'])
tmsg = mk_tool_res_msg([ToolUse(id='call_1', name='simple_add', arguments={})], ['4'])
claude_mk_payload([tmsg], 'claude-sonnet-5')['msgs']

[Msg(role='tool', content=[ToolResult(raw=None, cache_control=None, id='call_1', name='simple_add', arguments={}, server=False, text='4')], raw=None)]

Claude Code sends thinking, text, and tool calls in separate messages. Each message restarts its content-block indices at zero. `_reidx` assigns indices across the whole response to keep those blocks separate in FastLLM's collector. The adapter then converts events to `Delta` objects and restores the original tool names.

In [ ]:
#| export
def _reidx():
    "Return a stateful function that indexes content blocks across messages"
    base,mx = 0,-1
    def f(ev):
        nonlocal base,mx
        t = ev.get('type')
        if t=='message_start': base,mx = base+mx+1,-1
        elif t in ('content_block_start','content_block_delta','content_block_stop') and 'index' in ev:
            mx = max(mx, ev['index'])
            ev = {**ev, 'index': ev['index']+base}
        return ev
    return f


In [ ]:
reindex = _reidx()
indices = [reindex(dict(type='content_block_start', index=0))['index'],
    reindex(dict(type='content_block_start', index=1))['index']]
reindex(dict(type='message_start'))
indices.append(reindex(dict(type='content_block_start', index=0))['index'])
test_eq(indices, [0,1,2])
indices

[0, 1, 2]

For a tool with no arguments, Claude Code sends an empty `partial_json` chunk. FastLLM's collector needs complete arguments before it yields the tool call. `_ToolBlocks` tracks tool blocks that have started and whether their JSON arguments have finished.

In [ ]:
#| export
class _ToolBlocks:
    def __init__(self): self.open,self.complete = {},set()

`observe` updates the tracking state after each normalized event. A tool delta ending in `}` marks its arguments complete. If a tool block closes without that marker, `observe` returns an extra delta containing `{}` as its arguments. It returns nothing extra for non-tool blocks or completed tool blocks.

In [ ]:
#| export
@patch
def observe(self:_ToolBlocks, ev, delta):
    et,idx = ev.get('type'),ev.get('index')
    if et=='content_block_start' and delta.tool_calls: self.open[idx] = delta.tool_calls[0]
    elif et=='content_block_delta' and delta.tool_calls and (nested_idx(ev, 'delta', 'partial_json') or '').endswith('}'):
        self.complete.add(idx)
    if et!='content_block_stop' or (call := self.open.pop(idx, None)) is None or idx in self.complete: return
    return Delta(tool_calls=[ToolUse(id=call.id, name=call.name, arguments={})], raw=dict(index=idx))

In [ ]:
blocks = _ToolBlocks()
empty_call = ToolUse(id='call_0', name='ping', arguments={})
test_eq(blocks.observe(dict(type='content_block_start', index=0), Delta(tool_calls=[empty_call])), None)
empty_delta = blocks.observe(dict(type='content_block_stop', index=0), Delta())
test_eq(empty_delta.tool_calls[0].arguments, {})
empty_delta

Delta(text='', thinking='', refusal='', tool_calls=[ToolUse(raw=None, cache_control=None, id='call_0', name='ping', arguments={}, server=False, text=None)], citations=[], finish_reason=None, usage=None, raw={'index': 0})

`_claude_deltas` selects `stream_event` messages from the raw run. It applies `_reidx` and Anthropic's `norm_sse_event` before yielding each delta. FastLLM's standard collector builds the response from those deltas.

Calls to the chat's own tools keep `ToolUse.server=False`. FastLLM's chat loop must execute these tools itself.

In [ ]:
#| export
async def _claude_deltas(run):
    "Normalize one ClaudeRun turn to FastLLM deltas"
    reindex = _reidx()
    blocks = _ToolBlocks()
    async for m in run:
        if m.get('type')!='stream_event': continue
        ev = reindex(m['event'])
        d = norm_sse_event(ev)
        for tc in (d.tool_calls or []): tc.name = unqual(tc.name)
        yield d
        if extra := blocks.observe(ev, d): yield extra

Claude Code can report a provider failure in its terminal result without raising an exception during iteration. After collecting the stream, `_check_run` raises FastLLM's `APIError` when that result has `is_error=True`. The exception includes the model, API error status, and raw result.

In [ ]:
#| export
def _check_run(run, payload):
    if not run.result or not run.result.get('is_error'): return
    raise APIError(str(run.result.get('result') or run.result.get('subtype')), provider='claude_code',
        model=payload.get('model'), status_code=run.result.get('api_error_status'), raw=run.result)

`claude_acollect_stream` starts the run and collects `_claude_deltas` with FastLLM's standard collector. It then checks the terminal result for errors. If the host cannot find the `claude` executable, it converts `FileNotFoundError` to a non-retryable `APIError`. Callers can handle both failures through FastLLM's provider error interface.

`ClaudeRun` performs cleanup when its event iterator exits. Breaking out of the outer stream early does not guarantee that cleanup has finished.

In [ ]:
#| export
async def claude_acollect_stream(payload, **kwargs):
    "Adapt one `ClaudeRun` to FastLLM"
    run = astream(**payload)
    try:
        async for o in mk_acollect_stream(_claude_deltas(run), index_fn=delta_index_fn, api_name='claude_code', **kwargs): yield o
    except FileNotFoundError as e: raise APIError(str(e), provider='claude_code', model=payload.get('model'), retryable=False) from e
    _check_run(run, payload)

In [ ]:
p = claude_mk_payload(mk_msgs(['hi']), 'claude-sonnet-5')
with expect_fail(APIError, contains='not found'): [o async for o in claude_acollect_stream(dict(p, claude_path='/nonexistent/claude'))]

The provider uses FastLLM's Anthropic functions to normalize tool calls, message parts, finish reasons, and usage. It also uses Anthropic's costing functions. `collate_raw` reconstructs the assistant message in Anthropic's wire format from the streamed events. FastLLM includes that message when it passes the history back to `astream`.

The registration declares no continuation capability. FastLLM replays the full message history after each tool round, as it does for its Codex provider.

In [ ]:
#| export
api_registry.register('claude_code',
    norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage,
    finalize_usage=finalize_usage, collate_raw=collate_raw, mk_payload=claude_mk_payload, acollect_stream=claude_acollect_stream, cost=cost)

## Subscription tokens

Run `claude setup-token` to obtain a long-lived token for a Claude subscription. Claude Code reads it from `CLAUDE_CODE_OAUTH_TOKEN`. The [Claude Code authentication docs](https://code.claude.com/docs/en/authentication#generate-a-long-lived-token) specify a one-year lifetime.

A host serving multiple users can store an auth dictionary for each user and pass its token as `oauth_token`. `claude_auth` prepares that dictionary from a pasted token or an existing dictionary. It trims whitespace and checks the `sk-ant-oat` prefix without contacting Claude.

`claude_token` returns the stored token and `None` for refreshed auth. It does not refresh tokens, including when `force_refresh=True`. `claude_info` returns `None` for both plan and expiry because the stored dictionary contains neither. These functions use the same host-facing interface as FastLLM's Codex module.

In [ ]:
#| export
def claude_auth(auth):
    "Prepare an auth dictionary from a setup token or an existing auth dictionary"
    token = auth.get('token') if isinstance(auth, dict) else auth
    token = token.strip() if isinstance(token, str) else ''
    if not token.startswith('sk-ant-oat'): raise ValueError('Claude auth is the token from `claude setup-token`')
    return dict(token=token)

async def claude_token(auth, force_refresh=False):
    "Return `(token, None)` from stored auth without refreshing the token"
    return auth['token'], None

def claude_info(auth):
    "Return `None` for plan and expiry, which the stored token does not describe"
    return dict(plan=None, expires=None)

In [ ]:
test_eq(claude_auth(' sk-ant-oat01-abc '), dict(token='sk-ant-oat01-abc'))
test_eq(claude_auth(dict(token='sk-ant-oat01-abc')), dict(token='sk-ant-oat01-abc'))
with expect_fail(ValueError, 'setup-token'): claude_auth('sk-ant-api03-notoauth')
await claude_token(dict(token='sk-ant-oat01-abc'))

('sk-ant-oat01-abc', None)

## Live runs

This example uses the real Claude Code CLI and spends subscription tokens. It runs when `CLAUDE_OAUTH_TOKEN` or `CLAUDE_CODE_OAUTH_TOKEN` is set. The token passes to `AsyncChat` as `oauth_token`, as it would on a serving host.

Claude requests `simple_add` in the first response. FastLLM runs the function and sends the history ending in its result. FastClaude supplies that result through deferred continuation in a fresh process. The chat finishes with `10` and no response id.

In [ ]:
tok = os.getenv('CLAUDE_OAUTH_TOKEN') or os.getenv('CLAUDE_CODE_OAUTH_TOKEN')
if tok:
    chat = AsyncChat('claude_code/claude-sonnet-5', tools=[simple_add], oauth_token=tok)
    rs = await chat('What is 7+3? Use the tool, then answer with only the number.', stream=True, max_steps=3)
    parts = [o async for o in rs]
    test_eq([m.role for m in chat.hist], ['user','assistant','tool','assistant'])
    test_eq(chat.hist[-1].text, '10')
    assert chat.response_id is None
[type(o).__name__ for o in parts] if tok else None

['ToolUse', 'Completion', 'ToolResult', 'Refresh', 'Text', 'Completion']